In [94]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy

In [95]:
np.random.seed(0)

In [96]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.ones_like(priors)
    return priors / np.sum(priors)

In [97]:
def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + (i*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i)*1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [98]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [99]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.0
    for partition in partitions:
        acc_loss_p  = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [100]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [101]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))
    best_partition, best_loss = None, np.inf
    for partitions in partitions_set:
        acc_loss = evaluate_system(X, partitions, thresholds, priors, threshold_true, c)
        if acc_loss < best_loss:
            best_loss      = acc_loss
            best_partition = partitions
    return best_partition

In [102]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({-acc_loss:.4e}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)

def find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=False, eps=1e-9):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1
    
    X_eps = np.arange(-1/c, 0, 1e-3).round(5)

    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c) * np.sum(priors[a])
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c) * np.sum(priors[b])

        acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab]) * eps
        acc_loss_a_eps = evaluate_partition(X_eps, a, thresholds, priors, threshold_true, c) * np.sum(priors[a]) * eps
        acc_loss_b_eps = evaluate_partition(X_eps, b, thresholds, priors, threshold_true, c) * np.sum(priors[b]) * eps

        acc_loss_ab += acc_loss_ab_eps
        acc_loss_a += acc_loss_a_eps
        acc_loss_b += acc_loss_b_eps
        
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        ab = sorted(a + b)
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)

        acc_loss_a_eps = evaluate_partition(X_eps, a, thresholds, priors, threshold_true, c) * eps
        acc_loss_b_eps = evaluate_partition(X_eps, b, thresholds, priors, threshold_true, c) * eps
        acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * eps

        acc_loss_a += acc_loss_a_eps
        acc_loss_b += acc_loss_b_eps
        acc_loss_ab += acc_loss_ab_eps

        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    heapq.heappush(pq2, (acc_loss, (x_id, y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    p = P[p_id]
                    merged = sorted(ab + p)
                    acc_loss_merged = evaluate_partition(X, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged])
                    acc_loss_p = evaluate_partition(X, p, thresholds, priors, threshold_true, c) * np.sum(priors[p])
                    acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])

                    acc_loss_merged_eps = evaluate_partition(X_eps, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged]) * eps
                    acc_loss_p_eps = evaluate_partition(X_eps, p, thresholds, priors, threshold_true, c) * np.sum(priors[p]) * eps
                    acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab]) * eps

                    acc_loss_merged += acc_loss_merged_eps
                    acc_loss_p += acc_loss_p_eps
                    acc_loss_ab += acc_loss_ab_eps

                    gain = -(acc_loss_p + acc_loss_ab - acc_loss_merged)
                    heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())

In [103]:
def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [104]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)

In [105]:
df = pd.read_pickle("../results/grid_search_approx_ratio_best_n4_tiebreaking.pkl")

In [106]:
df["opt_len"] = df["partition_opt"].apply(lambda x: len(x))
df["greedy_len"] = df["partition_greedy"].apply(lambda x: len(x))

In [107]:
df[(df["r_mult"]>2) & (df["r_mult"]<2.5)]

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors,opt_len,greedy_len
545420,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.043164,0.097914,2.268440,0.054751,"[0.0, 0.2, 0.4, 1.0]","[0.32, 0.04, 0.2, 0.44]",2,2
551080,0.1,0.75,"[[1], [0, 2, 3]]","[[0], [1, 2, 3]]",0.046715,0.099990,2.140411,0.053275,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.1, 0.18, 0.42]",2,2
551090,0.1,0.75,"[[1], [0, 2, 3]]","[[0], [1, 2, 3]]",0.046715,0.099990,2.140411,0.053275,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.1, 0.16, 0.44]",2,2
551720,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.040924,0.099990,2.443315,0.059066,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.06, 0.2, 0.44]",2,2
552050,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.044952,0.098262,2.185960,0.053311,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.04, 0.22, 0.44]",2,2
...,...,...,...,...,...,...,...,...,...,...,...,...
3109071,0.2,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.092045,0.191001,2.075086,0.098956,"[0.2, 0.6, 0.8, 1.0]","[0.36, 0.1, 0.4, 0.14]",2,2
3109081,0.2,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.092045,0.191001,2.075086,0.098956,"[0.2, 0.6, 0.8, 1.0]","[0.36, 0.1, 0.38, 0.16]",2,2
3109091,0.2,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.092045,0.191001,2.075086,0.098956,"[0.2, 0.6, 0.8, 1.0]","[0.36, 0.1, 0.36, 0.18]",2,2
3109101,0.2,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.092045,0.191001,2.075086,0.098956,"[0.2, 0.6, 0.8, 1.0]","[0.36, 0.1, 0.34, 0.2]",2,2


In [108]:
df_bad = df
df_bad = df_bad.sort_values("r_mult", ascending=False).reset_index(drop=True)
df_bad.head()

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors,opt_len,greedy_len
0,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.035382,0.098422,2.781665,0.063040,"[0.0, 0.2, 0.6, 1.0]","[0.32, 0.02, 0.36, 0.3]",2,2
1,0.1,0.75,"[[0, 1, 2, 3]]","[[1], [3], [0, 2]]",0.036484,0.093775,2.570270,0.057290,"[0.0, 0.4, 0.8, 1.0]","[0.32, 0.22, 0.28, 0.18]",1,3
2,0.1,0.75,"[[1], [0, 2, 3]]","[[0], [1, 2, 3]]",0.038926,0.099990,2.568713,0.061064,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.08, 0.18, 0.44]",2,2
3,0.1,0.75,"[[1], [0, 2, 3]]","[[3], [0, 1, 2]]",0.038926,0.099990,2.568713,0.061064,"[0.0, 0.6, 0.8, 1.0]","[0.3, 0.08, 0.48, 0.14]",2,2
4,0.1,0.75,"[[1], [0, 2, 3]]","[[3], [0, 1, 2]]",0.038926,0.099990,2.568713,0.061064,"[0.0, 0.2, 0.8, 1.0]","[0.3, 0.08, 0.48, 0.14]",2,2


In [ ]:
i = df_bad["r_mult"].argmax()
df_bad.iloc[[i]]

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors,opt_len,greedy_len
0,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.035382,0.098422,2.781665,0.06304,"[0.0, 0.2, 0.6, 1.0]","[0.32, 0.02, 0.36, 0.3]",2,2


In [276]:
thresholds = df_bad["thresholds"].iloc[i]
priors = df_bad["priors"].iloc[i]
threshold_true = df_bad["threshold_true"].iloc[i]
c = df_bad["c"].iloc[i]
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=True)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)
r

[  (1.5678e-03, ([0], [1]))  (1.4393e-10, ([0], [2]))  (1.0795e-10, ([2], [3]))  (5.9970e-12, ([1], [2]))  (1.1994e-11, ([1], [3]))  (-4.6465e-02, ([0], [3]))  ]
[  (1.0795e-10, ([2], [3]))  (-1.5678e-03, ([0, 1], [2]))  (-5.3261e-02, ([0, 1], [3]))  ]
[  (-8.6097e-02, ([2, 3], [0, 1]))  ]


np.float64(2.7816650652800545)

In [ ]:
eps = 0.00
thresholds = np.array([0., 0.2, 0.6, 1.0])
priors = np.array([0.32-eps, 0.02+eps, 0.36, 0.3])
print(np.sum(priors))
threshold_true = 0.1
c = 0.75

partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=True)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)
print()
print(f"Accuracy loss  : {loss_greedy:.6f}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print(f"r              : {r}")

1.0
[  (1.5678e-03, ([0], [1]))  (1.4393e-10, ([0], [2]))  (1.0795e-10, ([2], [3]))  (5.9970e-12, ([1], [2]))  (1.1994e-11, ([1], [3]))  (-4.6465e-02, ([0], [3]))  ]
[  (1.0795e-10, ([2], [3]))  (-1.5678e-03, ([0, 1], [2]))  (-5.3261e-02, ([0, 1], [3]))  ]
[  (-8.6097e-02, ([2, 3], [0, 1]))  ]

Accuracy loss  : 0.098422
Accuracy loss  : 0.035382
r              : 2.7816650652800545


In [278]:
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r}")
print("-"*80)
print()

,0,1,2,3
Thresholds,0.00,0.20,0.60,1.0
Priors,0.32,0.02,0.36,0.3


c              : 0.75
t*             : 0.1000

Greedy
------
Partition      : [[0, 1], [2, 3]]
Accuracy loss  : 0.098422

Optimal
-------
Partition      : [[1], [0, 2, 3]]
Accuracy loss  : 0.035382

r (mult)       : 2.7816650652800545
--------------------------------------------------------------------------------



In [ ]:
a, b = [0], [1]

acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors[ab])
gain = lhs - rhs

print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"              gain: {gain}")
print(f"            merge?: {lhs - rhs > -1e-6}")
print()

    threshold true: 0.1000
                 c: 0.75
                 a: [0]
                 b: [1]
   accuracy loss a: 0.0999900
   accuracy loss b: 0.0999900
  accuracy loss ab: 0.1002605
               LHS: 0.0339966
               RHS: 0.0340886
              gain: -9.197080291970666e-05
            merge?: False



In [443]:
p = [2,3]
# p = [1]
eps = 0.0023
priors = np.array([0.32-eps, 0.02+eps, 0.36, 0.3])
X_p = best_response_vectorized(X, thresholds[p], priors[p], c)
fig = px.scatter(x=X, y=X_p, labels={"x": "X", "y": "best response"}).update_layout(
    width=500, height=400,
    font=dict(family="iosevka"),
    margin=dict(t=30, b=30, l=30, r=30),
    yaxis=dict(range=(-0.05, 1.05))
    )

m = X[X_p!=X][1]
fig.add_vline(x=m, annotation=dict(text=f"{m:.4f}"))
print(f"{thresholds[1]-m:.4f}")
fig

0.2872


### Effect of epsilon and delta

We redistribute the priors of classifiers 0 and 1 by taking $\epsilon$ from 0 and adding it to 1. 

We also add $\delta$ to the threshold of classifier 1. 

In [ ]:
epsilons = np.arange(0, 0.003+1e-4, 1e-4).round(4)
deltas = np.arange(0.08, 0.1, 1e-4).round(4)
results_eps = {"epsilon": [], "delta": [], "threshold": [], "priors": [], "partition_greedy": [], "partition_opt": [], "loss_greedy": [], "loss_opt": [], "r": []}
ec = 0
for eps in tqdm.tqdm(epsilons):
    # priors_eps =  np.array([0.32/(1-eps), 0.02-eps, 0.36/(1-eps), 0.3/(1-eps)])
    # priors_eps =  np.array([0.32 - (0.32 * eps/0.98), 0.02+eps, 0.36 - (0.36 * eps/0.98), 0.3 - (0.3 * eps/0.98)])
    priors_eps = np.array([0.32-eps, 0.02+eps, 0.36, 0.3])
    # priors_eps = np.array([0.32, 0.02, 0.36+eps, 0.3-eps])
    for delta in deltas:
        thresholds = np.array([0., 0.2+delta, 0.6, 1.0])
        partition_opt = find_partitions_optimal(X, thresholds, priors_eps, threshold_true, c)
        partition_greedy = find_partitions_greedy_best(X, thresholds, priors_eps, threshold_true, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors_eps, threshold_true, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_eps, threshold_true, c)
        r = approximation_ratio(loss_opt, loss_greedy)
        
        results_eps["epsilon"].append(eps)
        results_eps["delta"].append(delta)
        results_eps["threshold"].append(thresholds)
        results_eps["priors"].append(priors_eps)
        results_eps["partition_greedy"].append(f"{partition_greedy}")
        results_eps["partition_opt"].append(f"{partition_opt}")
        results_eps["loss_greedy"].append(loss_greedy)
        results_eps["loss_opt"].append(loss_opt)
        results_eps["r"].append(r.item())

df_eps = pd.DataFrame(results_eps)

100%|██████████| 31/31 [03:18<00:00,  6.42s/it]


In [325]:
i_max = df_eps["r"].argmax()
eps_max = df_eps["epsilon"].iloc[i_max]
delta_max = df_eps["delta"].iloc[i_max]

fig = px.scatter(df_eps, x="epsilon", y="delta", color="r", hover_data=["partition_greedy", "partition_opt"]).update_layout(
    width=500, height=400,
    font=dict(family="iosevka"),
    margin=dict(t=30, b=30, l=30, r=30),
    )
# fig.add_vline(x=eps_max, annotation=dict(text=f"{eps_max:.4f}"))
fig

In [333]:
eps_max, delta_max

(np.float64(0.0023), np.float64(0.0873))

In [437]:
priors_eps = np.array([0.32-eps_max, 0.02+eps_max, 0.36, 0.3])
thresholds = np.array([0.0, 0.2+delta_max, 0.55, 0.958])
partition_opt = find_partitions_optimal(X, thresholds, priors_eps, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors_eps, threshold_true, c, True)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors_eps, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_eps, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors_eps}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r}")
print("-"*80)
print()

[  (2.2298e-06, ([0], [1]))  (1.3099e-10, ([0], [2]))  (1.1010e-10, ([2], [3]))  (4.3798e-12, ([1], [2]))  (1.1200e-11, ([1], [3]))  (-3.3147e-02, ([0], [3]))  ]
[  (1.1010e-10, ([2], [3]))  (-2.2297e-06, ([0, 1], [2]))  (-4.0840e-02, ([0, 1], [3]))  ]
[  (-9.2016e-02, ([2, 3], [0, 1]))  ]


,0,1,2,3
Thresholds,0.0000,0.2873,0.55,0.958
Priors,0.3177,0.0223,0.36,0.300


c              : 0.75
t*             : 0.1000

Greedy
------
Partition      : [[0, 1], [2, 3]]
Accuracy loss  : 0.099988

Optimal
-------
Partition      : [[1], [0, 2, 3]]
Accuracy loss  : 0.061714

r (mult)       : 1.6201842190537914
--------------------------------------------------------------------------------



2.941110882352942

In [395]:
X_eps = np.arange(-1/c, 0, 1e-3).round(5)

a, b = [1], [0,2,3]

priors_eps = np.array([0.32-eps_max, 0.02+eps_max, 0.36, 0.3])
# thresholds = np.array([0.09, 0.2+delta_max, 0.6, 1.0])

acc_loss_a = evaluate_partition(X, a, thresholds, priors_eps, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors_eps, threshold_true, c)

acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * 1e-6
acc_loss_a_eps = evaluate_partition(X_eps, a, thresholds, priors, threshold_true, c) * 1e-6
acc_loss_b_eps = evaluate_partition(X_eps, b, thresholds, priors, threshold_true, c) * 1e-6

acc_loss_ab += acc_loss_ab_eps
acc_loss_a += acc_loss_a_eps
acc_loss_b += acc_loss_b_eps

lhs = acc_loss_a * np.sum(priors_eps[a]) + acc_loss_b * np.sum(priors_eps[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors_eps, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors_eps[ab])
gain = lhs - rhs

print(f"               eps: {eps}")
print(f"            priors: {priors_eps}")
print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"      LHS ([a][b]): {lhs:.7f}")
print(f"       RHS ([a,b]): {rhs:.7f}")
print(f"              gain: {gain}")
print(f"            merge?: {gain > -1e-6}")
print()

               eps: 0.0023
            priors: [0.3177 0.0223 0.36   0.3   ]
    threshold true: 0.1000
                 c: 0.75
                 a: [1]
                 b: [0, 2, 3]
   accuracy loss a: 0.0999908
   accuracy loss b: 0.0999902
  accuracy loss ab: 0.0999900
      LHS ([a][b]): 0.0999902
       RHS ([a,b]): 0.0999900
              gain: 2.2006113004069405e-07
            merge?: True



In [410]:
p = [2,3]
# p = [1]
priors_eps = np.array([0.32-eps_max, 0.02+eps_max, 0.36, 0.3])
thresholds = np.array([0.0, 0.2+delta_max, 0.6, 0.9])
X_p = best_response_vectorized(X, thresholds[p], priors_eps[p], c)
fig = px.scatter(x=X, y=X_p, labels={"x": "X", "y": "best response"}).update_layout(
    width=500, height=400,
    font=dict(family="iosevka"),
    margin=dict(t=30, b=30, l=30, r=30),
    yaxis=dict(range=(-0.05, 1.05))
    )

m = X[X_p!=X][0]
fig.add_vline(x=m, annotation=dict(text=f"{m:.4f}"))
print(f"{thresholds[1]-m:.4f}")
fig

0.2873


### Effect of True Threshold

In [358]:
results_tt = {"threshold_true": [], "threshold": [], "priors": [], "partition_greedy": [], "partition_opt": [], "loss_greedy": [], "loss_opt": [], "r": []}
threshold_trues = np.arange(0.09, 0.11+1e-4, 1e-4).round(4)
for t_true in tqdm.tqdm(threshold_trues):
    priors_eps = np.array([0.32-eps_max, 0.02+eps_max, 0.36, 0.3])
    thresholds = np.array([0.0, 0.2+delta_max, 0.6, 1.0])
    partition_opt = find_partitions_optimal(X, thresholds, priors_eps, t_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors_eps, t_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors_eps, t_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_eps, t_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)

    results_tt["threshold_true"].append(t_true)
    results_tt["threshold"].append(thresholds)
    results_tt["priors"].append(priors_eps)
    results_tt["partition_greedy"].append(partition_greedy)
    results_tt["partition_opt"].append(partition_opt)
    results_tt["loss_greedy"].append(loss_greedy)
    results_tt["loss_opt"].append(loss_opt)
    results_tt["r"].append(r)

100%|██████████| 202/202 [00:06<00:00, 29.83it/s]


In [362]:
px.line(results_tt, x="threshold_true", y="r")\
    .update_layout(
        width=500, height=400,
        font=dict(family="iosevka"),
        margin=dict(t=30, b=30, l=30, r=30),
    )

In [369]:
epsilons = np.arange(0, 0.3+1e-2, 1e-2).round(2)
results_eps2 = {"epsilon": [], "threshold": [], "priors": [], "partition_greedy": [], "partition_opt": [], "loss_greedy": [], "loss_opt": [], "r": []}
ec = 0
for eps in tqdm.tqdm(epsilons):
    priors_eps = np.array([0.32-eps_max, 0.02+eps_max, 0.36-eps, 0.3+eps])
    thresholds = np.array([0., 0.2+delta_max, 0.6, 1.0])
    partition_opt = find_partitions_optimal(X, thresholds, priors_eps, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors_eps, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors_eps, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors_eps, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_eps2["epsilon"].append(eps)
    results_eps2["threshold"].append(thresholds)
    results_eps2["priors"].append(priors_eps)
    results_eps2["partition_greedy"].append(f"{partition_greedy}")
    results_eps2["partition_opt"].append(f"{partition_opt}")
    results_eps2["loss_greedy"].append(loss_greedy)
    results_eps2["loss_opt"].append(loss_opt)
    results_eps2["r"].append(r.item())

df_eps2 = pd.DataFrame(results_eps2)

100%|██████████| 31/31 [00:00<00:00, 32.15it/s]


In [371]:
px.line(df_eps2, x="epsilon", y="r")\
    .update_layout(
        width=500, height=400,
        font=dict(family="iosevka"),
        margin=dict(t=30, b=30, l=30, r=30),
    )

In [332]:
X_eps = np.arange(-1/c, 0, 1e-3).round(5)

a, b = [0], [1]

priors_eps = np.array([0.32-eps_max, 0.02+eps_max, 0.36, 0.3])
thresholds = np.array([0.09, 0.2+delta_max, 0.6, 1.0])

acc_loss_a = evaluate_partition(X, a, thresholds, priors_eps, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors_eps, threshold_true, c)

acc_loss_ab_eps = evaluate_partition(X_eps, ab, thresholds, priors, threshold_true, c) * 1e-6
acc_loss_a_eps = evaluate_partition(X_eps, a, thresholds, priors, threshold_true, c) * 1e-6
acc_loss_b_eps = evaluate_partition(X_eps, b, thresholds, priors, threshold_true, c) * 1e-6

acc_loss_ab += acc_loss_ab_eps
acc_loss_a += acc_loss_a_eps
acc_loss_b += acc_loss_b_eps

lhs = acc_loss_a * np.sum(priors_eps[a]) + acc_loss_b * np.sum(priors_eps[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors_eps, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors_eps[ab])
gain = lhs - rhs

print(f"               eps: {eps}")
print(f"            priors: {priors_eps}")
print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"      LHS ([a][b]): {lhs:.7f}")
print(f"       RHS ([a,b]): {rhs:.7f}")
print(f"              gain: {gain}")
print(f"            merge?: {gain > -1e-6}")
print()

               eps: 0.0023
            priors: [0.3177 0.0223 0.36   0.3   ]
    threshold true: 0.1000
                 c: 0.75
                 a: [0]
                 b: [1]
   accuracy loss a: 0.0999909
   accuracy loss b: 0.0999908
  accuracy loss ab: 0.0999834
      LHS ([a][b]): 0.0339969
       RHS ([a,b]): 0.0339944
              gain: 2.543528596507527e-06
            merge?: True



# Adding dummy 0s

### 1 dummy 0

In [340]:
thresholds = np.array([0., 0., 0.2+delta_max, 0.6, 1.])
threshold_true = 0.1
c = 0.75
epsilons = np.arange(0., 0.16+1e-3, 1e-3).round(3)
results_eps = {"epsilon": [], "prior": [], "partition_greedy": [], "partition_opt": [], "loss_greedy": [], "loss_opt": [], "r": []}
for eps in tqdm.tqdm(epsilons):
    priors = np.array([0.32-eps_max-eps, eps, 0.02+eps_max, 0.36, 0.3])
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_eps["prior"].append(priors)
    results_eps["epsilon"].append(eps)
    results_eps["partition_greedy"].append(partition_greedy)
    results_eps["partition_opt"].append(partition_opt)
    results_eps["loss_greedy"].append(loss_greedy)
    results_eps["loss_opt"].append(loss_opt)
    results_eps["r"].append(r)

100%|██████████| 161/161 [00:15<00:00, 10.45it/s]


In [343]:
df_res_eps = pd.DataFrame(results_eps)
df_res_eps[["partition_greedy", "partition_opt", "prior"]] = df_res_eps[["partition_greedy", "partition_opt", "prior"]].astype(str)

In [346]:
px.line(df_res_eps, x="epsilon", y="r", hover_data=["partition_greedy", "partition_opt"]).update_layout(
    width=500, height=400,
    font=dict(family="iosevka"),
    margin=dict(t=30, b=30, l=30, r=30),
    )

In [345]:
zero_eps_max = epsilons[np.argmax(results_eps["r"])]
priors = np.array([0.32-eps_max-zero_eps_max, zero_eps_max, 0.02+eps_max, 0.36, 0.3])
thresholds = np.array([0., 0., 0.2+delta_max, 0.6, 1.])
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r}")
print("-"*80)
print()

,0,1,2,3,4
Thresholds,0.0000,0.0,0.2873,0.60,1.0
Priors,0.3177,0.0,0.0223,0.36,0.3


c              : 0.75
t*             : 0.1000

Greedy
------
Partition      : [[0, 2], [1, 3, 4]]
Accuracy loss  : 0.099988

Optimal
-------
Partition      : [[1, 2], [0, 3, 4]]
Accuracy loss  : 0.033997

r (mult)       : 2.941110882352942
--------------------------------------------------------------------------------



### 2 dummy 0s

In [75]:
def generate_prior_grid(n_components=3, n_balls=50, value=1):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls):
        counts = np.bincount(combo, minlength=n_components).round(5)
        grids.append(counts * step * value)
    return grids

In [76]:
thresholds = np.array([0., 0., 0., 0.2, 0.6, 1.])
threshold_true = 0.1
c = 0.75
priors_1_grid = generate_prior_grid(3, 50, 0.32)
results = {"prior": [], "partition_greedy": [], "partition_opt": [], "loss_greedy": [], "loss_opt": [], "r": []}
for priors_1 in tqdm.tqdm(priors_1_grid):
    priors = np.hstack((priors_1, np.array([0.02, 0.36, 0.3]))).round(5)
    # priors = np.array([0.32-eps, eps, 0.02, 0.36, 0.3])
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results["prior"].append(priors)
    results["partition_greedy"].append(partition_greedy)
    results["partition_opt"].append(partition_opt)
    results["loss_greedy"].append(loss_greedy)
    results["loss_opt"].append(loss_opt)
    results["r"].append(r)

100%|██████████| 1326/1326 [07:19<00:00,  3.01it/s]


In [86]:
df_res2 = pd.DataFrame(results)
df_res2[["partition_greedy", "partition_opt", "prior"]] = df_res2[["partition_greedy", "partition_opt", "prior"]].astype(str)

In [93]:
df_res2.sort_values(["r"], ascending=False)

,prior,partition_greedy,partition_opt,loss_greedy,loss_opt,r
0,[0.32 0. 0. 0.02 0.36 0.3 ],"[[1, 4, 5], [0, 2, 3]]","[[1, 2, 3], [0, 4, 5]]",0.098422,0.035382,2.781665
1275,[0. 0.32 0. 0.02 0.36 0.3 ],"[[0, 4, 5], [1, 2, 3]]","[[0, 2, 3], [1, 4, 5]]",0.098422,0.035382,2.781665
1325,[0. 0. 0.32 0.02 0.36 0.3 ],"[[0, 4, 5], [1, 2, 3]]","[[0, 1, 3], [2, 4, 5]]",0.098422,0.035382,2.781665
663,[0.096 0.0128 0.2112 0.02 0.36 0.3 ],"[[1, 2, 3], [0, 4, 5]]","[[3], [0, 1, 2, 4, 5]]",0.098174,0.035382,2.774657
1096,[0.0256 0.1984 0.096 0.02 0.36 0.3 ],"[[0, 1, 3], [2, 4, 5]]","[[3], [0, 1, 2, 4, 5]]",0.098174,0.035382,2.774657
...,...,...,...,...,...,...
512,[0.1216 0.096 0.1024 0.02 0.36 0.3 ],"[[3], [0, 1, 2, 4, 5]]","[[3], [0, 1, 2, 4, 5]]",0.035382,0.035382,1.000000
511,[0.1216 0.1024 0.096 0.02 0.36 0.3 ],"[[3], [0, 1, 2, 4, 5]]","[[3], [0, 1, 2, 4, 5]]",0.035382,0.035382,1.000000
510,[0.1216 0.1088 0.0896 0.02 0.36 0.3 ],"[[3], [0, 1, 2, 4, 5]]","[[3], [0, 1, 2, 4, 5]]",0.035382,0.035382,1.000000
482,[0.128 0.0832 0.1088 0.02 0.36 0.3 ],"[[3], [0, 1, 2, 4, 5]]","[[3], [0, 1, 2, 4, 5]]",0.035382,0.035382,1.000000


In [380]:
px.scatter(df_res2.sort_values(["r"], ascending=False), y="r", hover_data=["partition_greedy", "partition_opt"]).update_layout(
    width=500, height=400,
    font=dict(family="iosevka"),
    margin=dict(t=30, b=30, l=30, r=30),
    )

In [383]:
thresholds = np.array([0., 0.,  0.2, 0.6, 1.])
priors = np.array([0.3177, 0.0023,  0.02, 0.36, 0.3])
# priors = np.array([0.32, 0., 0., 0.02, 0.36, 0.3])
threshold_true = 0.1
c = 0.75
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, True)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

index = ["0", "new1", "1", "2", "3"] if len(thresholds) == 5 else ["0", "new1", "new2", "1", "2", "3"]
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}, index=index).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r}")
print("-"*80)
print()

[  (1.5778e-03, ([0], [2]))  (1.0795e-10, ([3], [4]))  (1.4289e-10, ([0], [3]))  (5.9970e-12, ([2], [3]))  (1.1994e-11, ([2], [4]))  (1.0345e-12, ([1], [3]))  (1.7241e-12, ([1], [4]))  (-4.5745e-02, ([0], [4]))  (3.4483e-13, ([1], [2]))  (6.9389e-18, ([0], [1]))  ]
[  (1.0795e-10, ([3], [4]))  (1.7241e-12, ([1], [4]))  (1.0345e-12, ([1], [3]))  (-9.9990e-06, ([0, 2], [1]))  (-1.5778e-03, ([0, 2], [3]))  (-5.2577e-02, ([0, 2], [4]))  ]
[  (6.0500e-02, ([3, 4], [0, 2]))  (-9.9990e-06, ([0, 2], [1]))  (2.2206e-12, ([3, 4], [1]))  ]
[  (-1.4661e-01, ([0, 2, 3, 4], [1]))  ]


,0,new1,1,2,3
Thresholds,0.0000,0.0000,0.20,0.60,1.0
Priors,0.3177,0.0023,0.02,0.36,0.3


c              : 0.75
t*             : 0.1000

Greedy
------
Partition      : [[1], [0, 2, 3, 4]]
Accuracy loss  : 0.037912

Optimal
-------
Partition      : [[1, 2], [0, 3, 4]]
Accuracy loss  : 0.033997

r (mult)       : 1.1151764705882357
--------------------------------------------------------------------------------



In [168]:
a, b = [0,2], [3,4]

acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors[ab])
gain = lhs - rhs

print(f"               eps: {eps}")
print(f"            priors: {priors}")
print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"      LHS ([a][b]): {lhs:.7f}")
print(f"       RHS ([a,b]): {rhs:.7f}")
print(f"              gain: {gain}")
print(f"            merge?: {gain > -1e-6}")
print()

               eps: 0.0023
            priors: [0.3177 0.0023 0.02   0.36   0.3   ]
    threshold true: 0.1000
                 c: 0.75
                 a: [0, 2]
                 b: [3, 4]
   accuracy loss a: 0.0953177
   accuracy loss b: 0.0999900
  accuracy loss ab: 0.0377691
      LHS ([a][b]): 0.0981822
       RHS ([a,b]): 0.0376822
              gain: 0.06049995000499949
            merge?: True

